In [1]:
from transformers import AutoConfig, AutoModelForImageClassification
import torch

from hf_utils import convert_hf_swinv2_state_dict
from swin2_utils import SwinV2Cfg, MySwinV2

In [2]:
hf_name = "microsoft/swinv2-tiny-patch4-window16-256"
hf_cfg = AutoConfig.from_pretrained(hf_name)
hf_model = AutoModelForImageClassification.from_pretrained(hf_name)
hf_state = hf_model.state_dict()

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

In [3]:
my_cfg = SwinV2Cfg(
    image_size=hf_cfg.image_size,
    patch_size=hf_cfg.patch_size,
    num_channels=hf_cfg.num_channels,
    embed_dim=hf_cfg.embed_dim,
    depths=tuple(hf_cfg.depths),
    num_heads=tuple(hf_cfg.num_heads),
    window_size=hf_cfg.window_size,
    pretrained_window_sizes=tuple(hf_cfg.pretrained_window_sizes),
    mlp_ratio=hf_cfg.mlp_ratio,
    qkv_bias=hf_cfg.qkv_bias,
    hidden_dropout_prob=hf_cfg.hidden_dropout_prob,
    attention_probs_dropout_prob=hf_cfg.attention_probs_dropout_prob,
    drop_path_rate=hf_cfg.drop_path_rate,
    layer_norm_eps=hf_cfg.layer_norm_eps,
    num_labels=hf_cfg.num_labels,
)
my_model = MySwinV2(my_cfg, num_classes=my_cfg.num_labels)
my_model = MySwinV2(my_cfg, num_classes=my_cfg.num_labels)

converted_state = convert_hf_swinv2_state_dict(hf_state, my_model)

missing, unexpected = my_model.load_state_dict(converted_state, strict=False)

# Sanity check

In [5]:
my_model.eval()
hf_model.eval()

x = torch.randn(2, 3, 256, 256)

with torch.no_grad():
    y_hf = hf_model(pixel_values=x).logits
    y_my = my_model(x)

print("max abs diff:", (y_hf - y_my).abs().max().item())
print("mean abs diff:", (y_hf - y_my).abs().mean().item())

max abs diff: 0.0
mean abs diff: 0.0
